In [0]:
%sql
-- ============================================================
--  Exploración Inicial y Análisisde Calidad
-- ============================================================

-- ============================================================
-- ¿Cuántos registros tenemos?
-- ============================================================

select count(*) as total_registros
from real_estate.bronze.properties_bronze



In [0]:
%sql
-- ============================================================
-- Explorar estructura de la tabla
-- ============================================================


describe table real_estate.bronze.properties_bronze

In [0]:
%sql
-- ============================================================
-- Ver muestra de datos (columnas principales)
-- ============================================================

select 
    id,
    ubicacion,
    precio,
    expensas,
    tipo_de_operacion,
    moneda,
    ambientes,
    metros_cuadrados_totales,
    metros_cuadrados_cubiertos,
    antiguedad,
    estado,
    zona
from real_estate.bronze.properties_bronze
limit 10;

In [0]:
%sql
-- ============================================================
-- EDA 2.1: Contar valores nulos por columna
-- ============================================================
-- Usamos CTE para calcular el total y los nulos por columna

with total as (
    select count(*) as n
    from real_estate.bronze.properties_bronze
),

nulos as (
    select
        count(*) - count(id) as nulos_id,
        count(*) - count(ubicacion) as nulos_ubicacion,
        count(*) - count(precio) as nulos_precios,
        count(*) - count(expensas) as nulos_expensas,
        count(*) - count(tipo_de_operacion) as nulos_tipo_operacion,
        count(*) - count(moneda) as nulos_moneda,
        count(*) - count(ambientes) as nulos_ambientes,
        count(*) - count(metros_cuadrados_totales) as nulos_m2_totales,
        count(*) - count(metros_cuadrados_cubiertos) as nulos_metros_m2_cubiertos,
        count(*) - count(antiguedad) as nulos_antiguedad,
        count(*) - count(estado) as nulos_estado,
        count(*) - count(zona) as nulos_zona
    from real_estate.bronze.properties_bronze

)
select 
    t.n as total_registros,
    n.*
from total t, nulos n


In [0]:
%sql
-- ============================================================
-- Porcentaje de nulos en columnas clave
-- ============================================================

with totales as (
    select count(*) as total
    from real_estate.bronze.properties_bronze
)
select 
    round((t.total - count(precio)) * 100.0 / t.total, 2) as pct_precios_nulos,
    round((t.total - count(expensas)) * 100.0 / t.total, 2) as pct_expensas_nulos,
    round((t.total - count(ambientes)) * 100.0 / t.total, 2) as pct_ambientes_nulos,
    ROUND((t.total - COUNT(metros_cuadrados_totales)) * 100.0 / t.total, 2) as pct_nulos_m2_totales,
    ROUND((t.total - COUNT(metros_cuadrados_cubiertos)) * 100.0 / t.total, 2) as pct_nulos_m2_cubiertos,
    ROUND((t.total - COUNT(orientacion_cardinal)) * 100.0 / t.total, 2) as pct_nulos_orientacion,
    ROUND((t.total - COUNT(antiguedad)) * 100.0 / t.total, 2) as pct_nulos_antiguedad
from real_estate.bronze.properties_bronze
cross join totales t 
group by t.total

In [0]:
%sql
-- ============================================================
-- Identificar columnas con >50% de valores nulos
-- ============================================================
-- Columnas con >50% nulos son críticas: hay que decidir si eliminarlas o imputarlas


with totales as (
    select count(*) as total
    from real_estate.bronze.properties_bronze
),
pct_nulos as (
select 
    round((t.total - count(precio)) * 100.0 / t.total, 2) as pct_precios_nulos,
    round((t.total - count(expensas)) * 100.0 / t.total, 2) as pct_expensas_nulos,
    round((t.total - count(ambientes)) * 100.0 / t.total, 2) as pct_ambientes_nulos,
    ROUND((t.total - COUNT(metros_cuadrados_totales)) * 100.0 / t.total, 2) as pct_nulos_m2_totales,
    ROUND((t.total - COUNT(metros_cuadrados_cubiertos)) * 100.0 / t.total, 2) as pct_nulos_m2_cubiertos,
    ROUND((t.total - COUNT(orientacion_cardinal)) * 100.0 / t.total, 2) as pct_nulos_orientacion,
    ROUND((t.total - COUNT(antiguedad)) * 100.0 / t.total, 2) as pct_nulos_antiguedad
from real_estate.bronze.properties_bronze
cross join totales t 
group by t.total
)
select columna, porcentaje_nulos
from pct_nulos 
unpivot  (
    porcentaje_nulos for columna in (pct_precios_nulos, pct_expensas_nulos, 
        pct_ambientes_nulos, 
        pct_nulos_m2_totales, pct_nulos_m2_cubiertos,
        pct_nulos_orientacion, pct_nulos_antiguedad
    )
)
where porcentaje_nulos > 50
order by porcentaje_nulos desc;

In [0]:
%sql
-- ============================================================
-- Distribución de tipo_de_operacion
-- ============================================================

select 
    tipo_de_operacion,
    count(*) as cantidad,
    round(count(*) * 100.0 / sum(count(*)) over (), 2) as porcentaje
from real_estate.bronze.properties_bronze
group by tipo_de_operacion
order by cantidad desc;

In [0]:
%sql
-- ============================================================
-- Distribución de moneda
-- ============================================================

select
    moneda,
    count(*) as cantidad,
    round(count(*) * 100.0 / sum(count(*)) over(), 2) as porcentaje
from real_estate.bronze.properties_bronze
group by moneda
order by cantidad desc
    

In [0]:
%sql
-- ============================================================
-- Distribución de ambientes
-- ============================================================

select 
    ambientes,
    count(*) as cantidad,
    round(count(*) * 100.0 / sum(count(*)) over(), 2) as porcentaje
from real_estate.bronze.properties_bronze
group by ambientes
order by ambientes;

In [0]:
%sql
-- ============================================================
-- Top 15 zonas con más propiedades
-- ============================================================

select 
    zona,
    count(*) as cantidad,
    round(count(*) * 100.0 / sum(count(*)) over(), 2) as porcentaje
from real_estate.bronze.properties_bronze
group by zona
order by cantidad desc
limit 15;

In [0]:
%sql
-- ============================================================
-- Distribución de estado de las propiedades
-- ============================================================

select 
    estado,
    count(*) as cantidad,
    round(count(*) * 100.0 / sum(count(*)) over(), 2) as porcentaje
from real_estate.bronze.properties_bronze
group by estado
order by cantidad desc;